# Demo 4.2: KV transfer one level up: the broader goal, with the next operation never written down

**Insight.** The goal heads re-read the goal from the prompt even when it names only the broader goal (an algorithm)
and the next operation has to be derived from it. Controlling what the goal heads read over the algorithm's name
makes the LM write a different algorithm, every line of it its own.

**Setup.** The prompt only says `Implement <algorithm> in <language>.` The source and destination prompts name two
algorithms that share a natural code prefix: here merge sort in Python and quick sort in C++, which both start with
the same base case. That prefix is prefilled up to the line where the two algorithms diverge; the indentation token
there is the pivot.

**Intervention (pivot-onwards KV transfer).** As in Demo 4, but the transplanted span is just the algorithm's name: at
the pivot, and for every subsequent token, the 250 goal heads read SRC's keys and values over "merge sort" instead of
DST's "quick sort". Everything else is untouched.

If the whole function the patched C++ run writes is a merge sort, the intervention is successful.

**Paper.** App. I (Figure 24, Table 8).

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (the paper's figure for this very pair):

<img src="figures/04.2_unspecified_op.png" width="700" alt="Broader-goal KV transfer: transplanting the goal heads' view of &quot;merge sort&quot; into a C++ quick-sort prompt makes the model write a C++ merge sort (App. I).">

*Broader-goal KV transfer (`fig:kv-goal`): (i) SRC, merge sort in Python; (ii) DST, quick sort in C++; (iii) DST with
the goal heads reading SRC's k/v over the algorithm name: a merge sort in C++, and the readout at the pivot moves too.*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()      # no generation-config warnings
huggingface_hub.utils.disable_progress_bars()

# as in the paper's figures: source red, destination green, patched run purple
from typing import Literal
ide_theme: Literal['dark', 'light'] = 'dark'
if ide_theme == 'dark':   # light shades; the task-span fill reuses the text color
    SRC_C, DST_C, PAT_C = 'salmon1', 'sea_green3', 'plum2'
    SRC_BG, DST_BG      = SRC_C, DST_C
    PIVOT_BG            = 'bright_yellow'
else:                     # the paper's colors (Figure 5)
    SRC_C, DST_C, PAT_C = '#cc0000', '#38761d', '#5b277d'
    SRC_BG, DST_BG      = '#f4cccc', '#d9ead3'
    PIVOT_BG            = '#ffe6b2'

repo root: /disk/u/arnab/Codes/Projects/goals


In [2]:
CUDA     = '7'
MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct'
G250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
C250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/control/control250__laterhalf.json'
PAIR     = 'merge_quick'   # two algorithms with a shared code prefix: merge sort, quick sort
FLIP     = False           # False: SRC = merge sort, DST = quick sort; True: the other way round
SRC_LANG, DST_LANG = 'python', 'cpp'
V_SCALE    = 4.0       # x the transplanted values read by the goal heads (paper: V x 4)
MAX_NEW    = 512       # greedy tokens per generation (the whole function)
RO_MAX_NEW = 64        # tokens per GoalScope readout

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
Qwen2.5-Coder-32B-Instruct, the 250 goal heads, and 250 random heads (none of them goal heads) as a control.

In [3]:
import json, torch
from rich.console import Console
from rich.text import Text
from rich.markup import escape

import src.nrgs.functional as FUNC
import src.nrgs.kv_transfer as PP
import src.nrgs.unspecified_data as U

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)

ctx = FUNC.setup(MODEL, device_map={'': 0})
load = lambda p: [tuple(h) for h in json.load(open(p))]
goal_heads, ctrl_heads = load(G250), load(C250)
ctx.gsp = None
print(f'goal heads: {len(goal_heads)} | control heads: {len(ctrl_heads)} (overlap {len(set(goal_heads) & set(ctrl_heads))})')

2026-11-03 20:12:55 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
goal heads: 250 | control heads: 250 (overlap 0)


## 1. The contrastive pair
The two prompts as the model sees them. The highlighted algorithm name is all we transplant (the shorter name is
padded with newlines, so the two line up token for token); the pivot is the last prefilled token.

In [4]:
src_alg, dst_alg = U.sides(PAIR, FLIP)
g = U.build_geometry(PAIR, FLIP, SRC_LANG, DST_LANG, ctx=ctx)

def show(tok, win, title, bg):
    ids = tok['input_ids'][0].tolist()
    start = max(i for i, t in enumerate(ids) if t == ctx.mt.tokenizer.convert_tokens_to_ids('<|im_start|>') and i < min(win))
    txt = Text()
    for i in range(start, len(ids)):
        s = ctx.mt.tokenizer.decode([ids[i]])
        pivot = i == len(ids) - 1                     # the pivot: the prompt's last token
        txt.append(s, style=f'bold black on {PIVOT_BG}' if pivot else f'black on {bg}' if i in win else 'dim')
    console.rule(title); console.print(txt)

show(g['src_tok'], set(g['src_w']), f'SRC prompt: {src_alg["name"]} in {SRC_LANG} (k/v donor)', SRC_BG)
show(g['dst_tok'], set(g['dst_w']), f'DST prompt: {dst_alg["name"]} in {DST_LANG} (the run we patch)', DST_BG)

─────────────────────────────────────────────── SRC prompt: merge sort in python (k/v donor) ───────────────────────────────────────────────

<|im_start|>user
Implement merge sort in Python. Name the function `func`.<|im_end|>
<|im_start|>assistant
<code>
def func(arr):
    if len(arr) <= 1:
        return arr
   

───────────────────────────────────────────── DST prompt: quick sort in cpp (the run we patch) ─────────────────────────────────────────────

<|im_start|>user
Implement quick sort in C++. Name the function `func`.<|im_end|>
<|im_start|>assistant
<code>
std::vector<int> func(std::vector<int> arr) {
    if (arr.size() <= 1) {
        return arr;
    }
   

## 2. Clean runs
Without any intervention, each prompt writes its own algorithm, and GoalScope at the pivot describes what the goal
heads encode there. An LM judge (Claude) reads the whole function and says which of the two algorithms it implements.

In [5]:
stub = lambda prompt: prompt.split('<code>\n')[-1]        # the prefilled code prefix, up to the pivot

def show_run(title, color, prompt, gen, readout=None, verdict=None):
    console.rule(f'[bold {color}]{title}[/]')
    if readout is not None:
        console.print(f'[bold]GoalScope at the pivot:[/] [italic]{escape(readout.strip())}[/]')
    console.print(f'[bold]continuation:[/]\n[grey50]{escape(stub(prompt))}[/][{color}]{escape(gen.rstrip())}[/]')
    if verdict: console.print(f'[bold]judge:[/] {verdict}')
    console.print()

def readout(tok, iface=None):
    return FUNC.goalscope_readout('', tok, tok['input_ids'].shape[1] - 1, heads=goal_heads,
                                  iface=iface, maxnew=RO_MAX_NEW, ctx=ctx)[0]['patched_gen']

name = {'SRC': f'[bold {SRC_C}]{src_alg["name"]} (SRC)[/]', 'DST': f'[bold {DST_C}]{dst_alg["name"]} (DST)[/]'}
def verdict(gen, lang, ro):
    code_v, ro_v = U.judge_code(PAIR, FLIP, gen, lang)['verdict'], U.judge_readout(PAIR, FLIP, ro)['verdict']
    return f'function implements {name.get(code_v, code_v)}; readout says {name.get(ro_v, ro_v)}'

body = lambda gen: gen.split('</code>')[0]                # the function, up to the closing tag
with FUNC.console.capture():                               # silence the per-readout log lines
    clean = {k: dict(gen=FUNC.greedy(g[f'{k}_tok'], max_new=MAX_NEW, ctx=ctx), ro=readout(g[f'{k}_tok'])) for k in ('src', 'dst')}
show_run(f'SRC run: {src_alg["name"]} in {SRC_LANG}', SRC_C, g['src_prompt'], body(clean['src']['gen']), clean['src']['ro'],
         verdict(clean['src']['gen'], SRC_LANG, clean['src']['ro']))
show_run(f'DST run: {dst_alg["name"]} in {DST_LANG}', DST_C, g['dst_prompt'], body(clean['dst']['gen']), clean['dst']['ro'],
         verdict(clean['dst']['gen'], DST_LANG, clean['dst']['ro']))

────────────────────────────────────────────────────── SRC run: merge sort in python ───────────────────────────────────────────────────────

GoalScope at the pivot: I'm not executing any code; I'm just preparing to divide the task into smaller parts and merge them back together. 
No action is taken at a single point; the process involves recursively splitting the list into halves until base cases (single-element 
lists) are reached, then merging those splits.

continuation:
def func(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left_half = func(arr[:mid])
    right_half = func(arr[mid:])
    return merge(left_half, right_half)

def merge(left, right):
    sorted_array = []
    left_index, right_index = 0, 0
    while left_index < len(left) and right_index < len(right):
        if left[left_index] < right[right_index]:
            sorted_array.append(left[left_index])
            left_index += 1
        else:
            sorted_array.append(right[right_index])
            right_index += 1
    sorted_array.extend(left[left_index:])
    sorted_array.extend(right[right_index:])
    return sorted_array

judge: function implements merge sort (SRC); readout says merge sort (SRC)

──────────────────────────────────────────────────────── DST run: quick sort in cpp ────────────────────────────────────────────────────────

GoalScope at the pivot: I'm not executing any code directly; I'm using a divide-and-conquer approach to sort the array. Here's a brief 
explanation:

1. Choose a pivot element.
2. Partition the array into two sub-arrays: elements less than the pivot and elements greater than the pivot.
3. Recursively apply the

continuation:
std::vector<int> func(std::vector<int> arr) {
    if (arr.size() <= 1) {
        return arr;
    }
    int pivot = arr[arr.size() / 2];
    std::vector<int> left, right, equal;
    for (int num : arr) {
        if (num < pivot) {
            left.push_back(num);
        } else if (num > pivot) {
            right.push_back(num);
        } else {
            equal.push_back(num);
        }
    }
    left = func(left);
    right = func(right);
    left.insert(left.end(), equal.begin(), equal.end());
    left.insert(left.end(), right.begin(), right.end());
    return left;
}

judge: function implements quick sort (DST); readout says quick sort (DST)

## 3. Patched runs
Two versions of the DST run, each with the name transfer from the pivot on:
- **goal 250, V × 4**: the paper's intervention.
- **control 250, V × 4**: the same swap through 250 random heads.

For each run we show the GoalScope readout at the patched pivot, the whole function, and the judge's verdicts: which
algorithm the function implements, and which one the readout describes.

In [6]:
arms = {'goal 250, V x4': goal_heads, 'control 250, V x4': ctrl_heads}
for arm, heads in arms.items():
    stack = PP.intervention_stack(g, goal_heads=heads, v_scale=V_SCALE, ctx=ctx)
    def iface(mt_, inputs, patches=None, model_kwargs=None, stack=stack):
        with stack():
            return mt_._model(**dict(inputs), **dict(model_kwargs or {}))
    with FUNC.console.capture():
        with stack():
            gen = FUNC.greedy(g['dst_tok'], max_new=MAX_NEW, ctx=ctx)
        ro = readout(g['dst_tok'], iface=iface)
    show_run(f'patched DST run: {arm}', PAT_C, g['dst_prompt'], body(gen), ro, verdict(gen, DST_LANG, ro))

───────────────────────────────────────────────────── patched DST run: goal 250, V x4 ──────────────────────────────────────────────────────

GoalScope at the pivot: I'm not executing any code directly; I'm here to help you by dividing your request into smaller parts, sorting them,
and then merging them back together in order. No actual coding is done by me directly—instead, I recursively split the task into halves, 
sort each half, and then merge the sorted halves.

continuation:
std::vector<int> func(std::vector<int> arr) {
    if (arr.size() <= 1) {
        return arr;
    }
    int mid = arr.size() / 2;
    std::vector<int> left(arr.begin(), arr.begin() + mid);
    std::vector<int> right(arr.begin() + mid, arr.end());
    left = func(left);
    right = func(right);
    arr.clear();
    while (!left.empty() || !right.empty()) {
        if (left.empty()) {
            arr.push_back(right.front());
            right.erase(right.begin());
        } else if (right.empty()) {
            arr.push_back(left.front());
            left.erase(left.begin());
        } else if (left.front() < right.front()) {
            arr.push_back(left.front());
            left.erase(left.begin());
        } else {
            arr.push_back(right.front());
            right.erase(right.begin());
        }
    }
    return arr;
}

judge: function implements merge sort (SRC); readout says merge sort (SRC)

──────────────────────────────────────────────────── patched DST run: control 250, V x4 ────────────────────────────────────────────────────

GoalScope at the pivot: I'm not executing any code directly; I'm using a divide-and-conquer approach to sort it. Here's a brief explanation:

1. Choose a pivot element.
2. Partition the array into two sub-arrays: elements less than the pivot and elements greater than the pivot.
3. Recursively apply the same

continuation:
std::vector<int> func(std::vector<int> arr) {
    if (arr.size() <= 1) {
        return arr;
    }
    int pivot = arr[arr.size() / 2];
    std::vector<int> left, middle, right;
    for (int num : arr) {
        if (num < pivot) {
            left.push_back(num);
        } else if (num == pivot) {
            middle.push_back(num);
        } else {
            right.push_back(num);
        }
    }
    left = func(left);
    right = func(right);
    left.insert(left.end(), middle.begin(), middle.end());
    left.insert(left.end(), right.begin(), right.end());
    return left;
}

judge: function implements quick sort (DST); readout says quick sort (DST)

## What to expect
In the patched run, the C++ function should implement merge sort instead of quick sort, although nothing in the
destination prompt besides the transplanted name points to it, and the readout at the pivot should move with it.
Across 96 instances (12 algorithm pairs × 2 directions × 4 language directions) the whole function switches to SRC's
algorithm 77% of the time (74 of 96):

| language direction | py → py | py → cpp | cpp → py | cpp → cpp |
|---|---|---|---|---|
| function implements SRC's algorithm | 19 / 24 (79%) | 18 / 24 (75%) | 19 / 24 (79%) | 18 / 24 (75%) |

See App. I (Figure 24, Table 8) in the paper; Figure 24 is this very pair.